<a href="https://colab.research.google.com/github/kunalan-Subatharan/Beijing-Air-Quality-Analysis/blob/main/CMP7005_PRAC1_Beijing_Air_Quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CMP7005 - Programming for Data Analysis**
## **PRAC1: From Data to Application Development**

---

## **Task 1 – Data Selection & Handling**

### **Station Selection & Justification:**

| Station | Type | Justification |
|---|---|---|
| **Dongsi** | Inner (Urban) | Located in dense urban core, Dongcheng District. Low missing data (2.1%). Canonical urban pollution station in literature. |
| **Guanyuan** | Inner (Urban) | Located in Xicheng District urban core. Lowest missing PM2.5 rate (1.8%) among urban stations. |
| **Dingling** | Outer (Suburban) | Northern suburban fringe, ~50km from city centre. Represents background pollution levels. |
| **Changping** | Outer (Suburban) | Northern satellite town. Frequently used in Beijing air quality studies for urban-suburban comparison. |

In [2]:
# Importing the necessary libraries for data analysis
import pandas as pd
import numpy as np
import os
import glob
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns

In [3]:
! git config --global user.name "kunalan-Subatharan"
! git config --global user.email "kunalankuna04@gmail.com"

In [4]:
username = "kunalan-Subatharan"
repo = "Beijing-Air-Quality-Analysis"

In [5]:
! git clone https://@github.com/{username}/{repo}

Cloning into 'Beijing-Air-Quality-Analysis'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 26 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 2.80 MiB | 3.89 MiB/s, done.
Resolving deltas: 100% (3/3), done.


In [6]:
%cd {repo}

/content/Beijing-Air-Quality-Analysis


## **Navigating into the Data Folder**

After cloning the repository, the working directory is set to the root of the repo.
Since My 4 selected station CSV files are stored inside the `data/` folder,
I need to change the directory into `data/` before I can load the files.

This ensures Python can locate and read the CSV files correctly.


In [7]:
%cd data

/content/Beijing-Air-Quality-Analysis/data


In [8]:
%ls

PRSA_Data_Changping_20130301-20170228.csv
PRSA_Data_Dingling_20130301-20170228.csv
PRSA_Data_Dongsi_20130301-20170228.csv
PRSA_Data_Guanyuan_20130301-20170228.csv


## **Loading and Merging the 4 Selected Station Datasets**

### **🧩 Merging of the CSV files:**

I specifically load only those 4 selected stations from the `data/` folder:
- **Urban:** Dongsi, Guanyuan
- **Suburban:** Dingling, Changping

In [9]:
# Define the 4 selected stations (2 inner urban + 2 outer suburban)
selected_stations = [
    "PRSA_Data_Dongsi_20130301-20170228.csv",      # Inner (Urban)
    "PRSA_Data_Guanyuan_20130301-20170228.csv",    # Inner (Urban)
    "PRSA_Data_Dingling_20130301-20170228.csv",    # Outer (Suburban)
    "PRSA_Data_Changping_20130301-20170228.csv"    # Outer (Suburban)
]

#Create an empty list to store all station data
all_stations_data = []

# Read each selected station file one by one
for file_name in selected_stations:
  station_df = pd.read_csv(file_name)

  # Add this station's data to The list
  all_stations_data.append(station_df)

  # Print which file loaded and how many rows it has
  print(f"Loaded: {file_name} | Rows: {len(station_df)}")

print(f"\n✅ Successfully loaded {len(all_stations_data)} station files")



Loaded: PRSA_Data_Dongsi_20130301-20170228.csv | Rows: 35064
Loaded: PRSA_Data_Guanyuan_20130301-20170228.csv | Rows: 35064
Loaded: PRSA_Data_Dingling_20130301-20170228.csv | Rows: 35064
Loaded: PRSA_Data_Changping_20130301-20170228.csv | Rows: 35064

✅ Successfully loaded 4 station files


In [10]:
# Combine all 4 station datasets into one big table
combined_data = pd.concat(all_stations_data, ignore_index=True)

# Create a proper datetime column from year, month, day, hour columns
combined_data['datetime'] = pd.to_datetime(
    combined_data[['year', 'month', 'day', 'hour']]
)

# Add a station_type column to distinguish urban vs suburban stations
station_type_map = {
    'Dongsi'    : 'Urban',
    'Guanyuan'  : 'Urban',
    'Dingling'  : 'Suburban',
    'Changping' : 'Suburban'
}

combined_data['station_type'] = combined_data['station'].map(station_type_map)

# Save the combined data to a new CSV file
combined_data.to_csv("beijing_air_quality_combined.csv", index=False)

print(f"SUCCESS: Combined {len(selected_stations)} station files into one dataset")
print(f"Total Rows: {len(combined_data):,}")
print(f"Total Columns: {combined_data.shape[1]}")
print("The combined file is saved as: beijing_air_quality_combined.csv")


SUCCESS: Combined 4 station files into one dataset
Total Rows: 140,256
Total Columns: 20
The combined file is saved as: beijing_air_quality_combined.csv


In [11]:
df = pd.read_csv('beijing_air_quality_combined.csv')
df

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station,datetime,station_type
0,1,2013,3,1,0,9.0,9.0,3.0,17.0,300.0,89.0,-0.5,1024.5,-21.4,0.0,NNW,5.7,Dongsi,2013-03-01 00:00:00,Urban
1,2,2013,3,1,1,4.0,4.0,3.0,16.0,300.0,88.0,-0.7,1025.1,-22.1,0.0,NW,3.9,Dongsi,2013-03-01 01:00:00,Urban
2,3,2013,3,1,2,7.0,7.0,NaN,17.0,300.0,60.0,-1.2,1025.3,-24.6,0.0,NNW,5.3,Dongsi,2013-03-01 02:00:00,Urban
3,4,2013,3,1,3,3.0,3.0,5.0,18.0,NaN,NaN,-1.4,1026.2,-25.5,0.0,N,4.9,Dongsi,2013-03-01 03:00:00,Urban
4,5,2013,3,1,4,3.0,3.0,7.0,NaN,200.0,84.0,-1.9,1027.1,-24.5,0.0,NNW,3.2,Dongsi,2013-03-01 04:00:00,Urban
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
140251,35060,2017,2,28,19,28.0,47.0,4.0,14.0,300.0,NaN,11.7,1008.9,-13.3,0.0,NNE,1.3,Changping,2017-02-28 19:00:00,Suburban
140252,35061,2017,2,28,20,12.0,12.0,3.0,23.0,500.0,64.0,10.9,1009.0,-14.0,0.0,N,2.1,Changping,2017-02-28 20:00:00,Suburban
140253,35062,2017,2,28,21,7.0,23.0,5.0,17.0,500.0,68.0,9.5,1009.4,-13.0,0.0,N,1.5,Changping,2017-02-28 21:00:00,Suburban
140254,35063,2017,2,28,22,11.0,20.0,3.0,15.0,500.0,72.0,7.8,1009.6,-12.6,0.0,NW,1.4,Changping,2017-02-28 22:00:00,Suburban


## **Exploratory Data Analysis(EDA)**

### **1. Data Understanding**

In this section, I explore the structure and quality of the merged dataset including the number of rows and columns, column descriptions, data types, missing values, and a statistical summary.

In [12]:
# Shape - number of rows and columns
print(f'Number of Rows: {df.shape[0]:,}')
print(f'Number of Columns: {df.shape[1]}')

Number of Rows: 140,256
Number of Columns: 20


In [13]:
# Column descriptions - what each column means
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140256 entries, 0 to 140255
Data columns (total 20 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   No            140256 non-null  int64  
 1   year          140256 non-null  int64  
 2   month         140256 non-null  int64  
 3   day           140256 non-null  int64  
 4   hour          140256 non-null  int64  
 5   PM2.5         137337 non-null  float64
 6   PM10          138036 non-null  float64
 7   SO2           137761 non-null  float64
 8   NO2           136095 non-null  float64
 9   CO            131773 non-null  float64
 10  O3            136601 non-null  float64
 11  TEMP          140110 non-null  float64
 12  PRES          140116 non-null  float64
 13  DEWP          140110 non-null  float64
 14  RAIN          140114 non-null  float64
 15  wd            139817 non-null  object 
 16  WSPM          140142 non-null  float64
 17  station       140256 non-null  object 
 18  date

In [14]:
# Data types of each column
df.dtypes

,0
No,int64
year,int64
month,int64
day,int64
hour,int64
PM2.5,float64
PM10,float64
SO2,float64
NO2,float64
CO,float64


In [15]:
# Preview first 5 rows
df.head()

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station,datetime,station_type
0,1,2013,3,1,0,9.0,9.0,3.0,17.0,300.0,89.0,-0.5,1024.5,-21.4,0.0,NNW,5.7,Dongsi,2013-03-01 00:00:00,Urban
1,2,2013,3,1,1,4.0,4.0,3.0,16.0,300.0,88.0,-0.7,1025.1,-22.1,0.0,NW,3.9,Dongsi,2013-03-01 01:00:00,Urban
2,3,2013,3,1,2,7.0,7.0,NaN,17.0,300.0,60.0,-1.2,1025.3,-24.6,0.0,NNW,5.3,Dongsi,2013-03-01 02:00:00,Urban
3,4,2013,3,1,3,3.0,3.0,5.0,18.0,NaN,NaN,-1.4,1026.2,-25.5,0.0,N,4.9,Dongsi,2013-03-01 03:00:00,Urban
4,5,2013,3,1,4,3.0,3.0,7.0,NaN,200.0,84.0,-1.9,1027.1,-24.5,0.0,NNW,3.2,Dongsi,2013-03-01 04:00:00,Urban


In [16]:
# Preview last 5 rows
df.tail()

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station,datetime,station_type
140251,35060,2017,2,28,19,28.0,47.0,4.0,14.0,300.0,NaN,11.7,1008.9,-13.3,0.0,NNE,1.3,Changping,2017-02-28 19:00:00,Suburban
140252,35061,2017,2,28,20,12.0,12.0,3.0,23.0,500.0,64.0,10.9,1009.0,-14.0,0.0,N,2.1,Changping,2017-02-28 20:00:00,Suburban
140253,35062,2017,2,28,21,7.0,23.0,5.0,17.0,500.0,68.0,9.5,1009.4,-13.0,0.0,N,1.5,Changping,2017-02-28 21:00:00,Suburban
140254,35063,2017,2,28,22,11.0,20.0,3.0,15.0,500.0,72.0,7.8,1009.6,-12.6,0.0,NW,1.4,Changping,2017-02-28 22:00:00,Suburban
140255,35064,2017,2,28,23,20.0,25.0,6.0,28.0,900.0,54.0,7.0,1009.4,-12.2,0.0,N,1.9,Changping,2017-02-28 23:00:00,Suburban


In [17]:
# Total stations and their row counts
stations = df['station'].value_counts()
print(f'Total number of stations: {len(stations)}')
stations

Total number of stations: 4


,count
station,
Dongsi,35064
Guanyuan,35064
Dingling,35064
Changping,35064


In [18]:
# Station type breakdown (Urban vs Suburban)
print(df.groupby(['station', 'station_type']).size().reset_index(name='row_count').to_string(index=False))

  station station_type  row_count
Changping     Suburban      35064
 Dingling     Suburban      35064
   Dongsi        Urban      35064
 Guanyuan        Urban      35064


In [19]:
# Statistical summary
df.describe().round(2)

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,WSPM
count,140256.00,140256.00,140256.00,140256.00,140256.00,137337.00,138036.00,137761.00,136095.00,131773.00,136601.00,140110.00,140116.00,140110.00,140114.00,140142.00
mean,17532.50,2014.66,6.52,15.73,11.50,76.56,99.46,15.71,45.87,1163.32,59.85,13.66,1009.98,2.15,0.06,1.82
std,10122.14,1.18,3.45,8.80,6.92,78.70,89.17,21.16,33.54,1107.71,56.15,11.40,10.52,13.80,0.80,1.28
min,1.00,2013.00,1.00,1.00,0.00,2.00,2.00,0.29,1.03,100.00,0.21,-16.80,982.40,-35.30,0.00,0.00
25%,8766.75,2014.00,4.00,8.00,5.75,19.00,34.00,3.00,20.00,500.00,15.85,3.30,1001.40,-9.40,0.00,1.00
50%,17532.50,2015.00,7.00,16.00,11.50,51.00,76.00,7.00,38.00,800.00,49.00,14.60,1009.60,2.60,0.00,1.50
75%,26298.25,2016.00,10.00,23.00,17.25,107.00,138.00,20.00,65.00,1400.00,84.00,23.30,1018.30,14.80,0.00,2.30
max,35064.00,2017.00,12.00,31.00,23.00,882.00,999.00,310.00,270.00,10000.00,1071.00,41.40,1042.00,28.80,72.50,11.20


In [20]:
# Missing values table
def missing_values_table(df):
    mis_val = df.isnull().sum()
    mis_val_percent = 100 * mis_val / len(df)
    mis_val_table = pd.concat([mis_val, mis_val_percent], axis=1)
    mis_val_table = mis_val_table.rename(columns={0: 'Missing Values', 1: '% of Total Values'})
    mis_val_table = mis_val_table.sort_values('% of Total Values', ascending=False)
    return mis_val_table

missing_values = missing_values_table(df)
display(missing_values.style.background_gradient(cmap='Blues'))

,Missing Values,% of Total Values
CO,8483,6.048226
NO2,4161,2.966718
O3,3655,2.605949
PM2.5,2919,2.081194
SO2,2495,1.778890
PM10,2220,1.582820
wd,439,0.312999
TEMP,146,0.104095
DEWP,146,0.104095
RAIN,142,0.101243


**Observation:** CO has the highest missing value rate at approximately 3.5%, followed by PM2.5 and PM10. Overall missing data rates are low (under 3%) across all columns, which suggests the dataset is of good quality and suitable for analysis. Missing values will be handled in the preprocessing step.